# Entrenamiento de las redes profundas en Google Colab (GPU)

Este notebook entrena la **CNN 1D** y el **Transformer** con GPU gratuita, usando exactamente el mismo
código y protocolo que el entrenamiento local (`scripts/05_entrenar_profundos.py`).

**Antes de empezar**
1. En Colab: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)**.
2. En tu Google Drive debe existir `MyDrive/cesnet_tls_year22/muestra_2pct.zip`, con la muestra del 2 %
   comprimida desde tu equipo (ver `scripts/README.md`).

Los resultados se guardan en Drive al final, porque el disco de Colab se borra al cerrar la sesión.

## 1. Comprobar la GPU

In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())
print("Modelo de GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ninguna (cambia el entorno a GPU)")

## 2. Conectar Google Drive (datos de entrada y copia de los resultados)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/cesnet_tls_year22"
!ls -lh {DRIVE}

## 3. Clonar el repositorio público del proyecto e instalar dependencias

In [ ]:
!git clone https://github.com/jairowjp/cesnet-tls-classification.git /content/proyecto
%cd /content/proyecto
!git log --oneline -5
# Colab ya trae PyTorch con soporte de GPU: se instala el resto sin reemplazarlo
!pip install -q "pandas<3" pyarrow xgboost statsmodels pyyaml

## 4. Colocar la muestra del 2 % donde la espera el código

In [ ]:
!mkdir -p data/processed
!unzip -q -o {DRIVE}/muestra_2pct.zip -d data/processed
!ls data/processed/muestra_2pct/*.parquet | wc -l   # deben ser 117

## 5. Entrenar la CNN 1D
Cada época muestra la pérdida y el F1 macro de validación; la mejor época queda marcada con ←.

In [ ]:
!python scripts/05_entrenar_profundos.py --modelo cnn1d

## 6. Entrenar el Transformer encoder ligero

In [ ]:
!python scripts/05_entrenar_profundos.py --modelo transformer

## 7. Guardar los resultados en Drive
Se copian las métricas y predicciones (`results/modelos/`) y los pesos entrenados (`models/`).
Después, en tu equipo, descomprime `resultados_profundos.zip` en la raíz del repositorio y ejecuta
`python scripts/04_comparar_modelos.py`.

In [ ]:
!zip -q -r resultados_profundos.zip results/modelos/cnn1d results/modelos/transformer models/cnn1d.pt models/transformer.pt
!cp resultados_profundos.zip {DRIVE}/
!ls -lh {DRIVE}/resultados_profundos.zip